# Ноутбук получения меша по маске сегментации

#### перед запуском ноутбука
1. поместите файл ноутбука рядом с актуальной версией SpineTool2.0, т.к. для генерации используются утилиты тула
2. активируйте виртуальную среду тула SpineTool2.0 т.к. скрипт использует зависимости проекта спайнтул
3. если дендритные участки на снимках для обработки меньше 1% всего снимка (хотите учесть даже небольшие участки дендрита), скорректируйте MIN_COMPONENT_SIZE_RATE - процент вокселей от общего числа, который считается корректной компонентой связности сегментации. по умолчанию 1%

In [ ]:
from datetime import datetime
from itertools import product
from json import dump
import math
import os

import numpy as np
from scipy.ndimage import binary_erosion, generate_binary_structure, binary_dilation
from skimage.measure import marching_cubes, label
import point_cloud_utils as pcu
from tifffile import imwrite, imread, imshow
from pathlib import Path

from CGAL.CGAL_Polyhedron_3 import Polyhedron_3
from CGAL.CGAL_Kernel import Point_3

from projects.segmentation.utils.data_processing.final_segmentation import _segmenation_data_to_v_f_vv
from projects.segmentation.utils.data_processing.mesh import _save_off, _try_repair_with_cgal
from projects.segmentation.utils.data_processing.segmentation_utils import (
    find_main_label,
    get_spine_meshes,
    hash_point,
    point_to_list,
)
from projects.segmentation.utils.project_info import FinalSegmentationData


MIN_COMPONENT_SIZE_RATE = 0.01


def extract_large_components(
    segmentation: np.ndarray,
    min_component_size: int,
) -> list[np.ndarray]:
    """
    Извлекает большие компоненты связности из сегментации.

    Компонента строится по объединению:
        label == 1 (дендрит)
        label == 2 (шипики)

    Возвращает список масок исходного размера.
    """

    binary_mask = segmentation > 0

    labels, num = label(
        binary_mask,
        connectivity=3,
        return_num=True,
    )

    components = []

    for component_id in range(1, num + 1):
        component_mask = labels == component_id

        if component_mask.sum() < min_component_size:
            continue

        component_segmentation = np.zeros_like(segmentation, dtype=np.uint8)
        component_segmentation[component_mask] = segmentation[component_mask]

        # проверяем наличие и дендрита, и шипиков
        unique_values = np.unique(component_segmentation)

        if 1 not in unique_values:
            continue

        if 2 not in unique_values:
            continue

        components.append(component_segmentation)

    return components


def voxel_to_mesh(data: np.ndarray, shape: tuple, scale: list, folder: str) -> tuple[Polyhedron_3, np.ndarray]:
    """returns surface_poly of mesh and updated voxel data"""
    data[data > 0] = 1
    labels, dendrite_label = find_main_label(data)
    del data

    mesh_base = np.zeros_like(labels, dtype=np.uint8)
    mesh_base[labels == dendrite_label] = 1
    del labels

    # делаем erosion->dilation, но пересекаем его с исходной маской чтобы не уменьшать объект
    mesh_base = (binary_dilation(
        binary_erosion(
            mesh_base, structure=generate_binary_structure(3, 1), border_value=1
        ),
        structure=generate_binary_structure(3, 1),
    ) + mesh_base).astype(np.uint8) 

    padded = np.pad(mesh_base, 1, mode="constant", constant_values=0)

    vertices, facets, _, _ = marching_cubes(
        padded.astype(np.float32),
        level=0.5,
        spacing=(float(scale[0]), float(scale[1]), float(scale[2])),
    )
    facets = facets[:, [0, 2, 1]]

    # компенсируем padding: сдвигаем координаты обратно
    vertices -= np.array([scale[0], scale[1], scale[2]], dtype=np.float32)

    if vertices.shape[0] < 10000:
        print(("Too small object"))
        return

    vertices = np.asarray(vertices, dtype=np.float32)
    facets = np.asarray(facets, dtype=np.int32)

    _, _, cf, nf = pcu.connected_components(vertices, facets)
    cf = np.asarray(cf).reshape(-1)
    nf = np.asarray(nf).reshape(-1)

    if nf.size > 0:
        comp_max = int(np.argmax(nf))
        total_faces = int(np.sum(nf))
        max_faces = int(nf[comp_max])

        if total_faces > 0 and max_faces / total_faces > 0.98:
            vertices, facets, _, _ = pcu.remove_unreferenced_mesh_vertices(
                vertices, facets[cf == comp_max]
            )

    mesh_file = (
        "/mesh_tmp_"
        + str(datetime.now()).replace(".", "_").replace(" ", "_").replace(":", "_")
        + ".off"
    )
    _save_off(vertices, facets, folder + mesh_file)

    surface_poly = Polyhedron_3(folder + mesh_file)
    try:
        os.remove(folder + mesh_file)
    except:
        ...

    _try_repair_with_cgal(surface_poly)

    return surface_poly, mesh_base


def reconstruct_surface(
    data: np.ndarray, shape: tuple, scale: list, folder: str, 
):
    with np.errstate(all="ignore"):
        _, shaft_label = find_main_label(data, True)

        data_ = np.zeros_like(data, dtype=np.uint8)
        data_[data > 0] = 1
        surface_poly, _ = voxel_to_mesh(data_, shape, (1, 1, 1), folder)

        size = 3
        box_coords = list(
            product(
                range(-size, size + 1),
                range(-size, size + 1),
                range(-size, size + 1),
            )
        )
        segmentation = set()
        for v in surface_poly.vertices():
            p = np.array(
                [
                    min(shape[0] - 1, max(round(v.point().x()), 0)),
                    min(shape[1] - 1, max(round(v.point().y()), 0)),
                    min(shape[2] - 1, max(round(v.point().z()), 0)),
                ],
                dtype=np.uint16,
            )
            v.set_point(
                Point_3(
                    v.point().x() * scale[0],
                    v.point().y() * scale[1],
                    v.point().z() * scale[2],
                )
            )
            box = box_coords + p
            filter1 = np.all(box >= (0, 0, 0), axis=1)
            filter2 = np.all(box < tuple(shape), axis=1)
            box = box[filter1 & filter2]
            if (
                len(
                    box[
                        np.array(
                            data[box[:, 0], box[:, 1], box[:, 2]] == shaft_label
                        )
                    ]
                )
                < 3
                * len(box[np.array(data[box[:, 0], box[:, 1], box[:, 2]] > 0)])
                / 4
            ):
                segmentation.add(hash_point(v.point()))

        spine_meshes = get_spine_meshes(surface_poly, segmentation)

        spines = {}
        average = []
        for i, spine in enumerate(spine_meshes):
            spine_vertices = np.ndarray((spine.size_of_vertices(), 3)).astype(
                np.uint16
            )
            cur_spine_points = set()
            for j, p in enumerate(spine.points()):
                spine_vertices[j, :] = point_to_list(p, shape, scale)
                cur_spine_points.add(hash_point(p))
            spines[i] = cur_spine_points
            average.append(np.average(spine_vertices, axis=0).tolist())
        mesh_v_f_vv, spines_indices = _segmenation_data_to_v_f_vv(
            surface_poly, spines, shape, scale
        )

        spines = {
            "spines": {},
            "pos_to_id": {},
        }
        spines_files = []
        spine_ids = list(spines_indices.keys())
        spine_ids.sort()
        files = {}
        for i, id in enumerate(spine_ids):
            filename = (
             f"/spine_{id}.off"
            )
            spines["spines"][id] = {
                "id": id,
                "pos": i,
                "average": average[id],
                "indices": spines_indices[id],
            }
            files[id] = filename
            spines["pos_to_id"][i] = id
            spines_files.append(filename)

        mesh_filename = (
            f"/surface_mesh.off"
        )
        surface_poly.write_to_file(folder + mesh_filename)

        for i, spine in enumerate(spine_meshes):
            spine.write_to_file(folder + files[i])

        additional_file = (
            "/spines_"
            + str(datetime.now())
            .replace(".", "_")
            .replace(" ", "_")
            .replace(":", "_")
            + ".json"
        )
        f = open(folder + additional_file, "w")
        dump(spines, f)
        f.close()

        return (
            (
                FinalSegmentationData(mesh_filename, mesh_v_f_vv, spines_files),
                {"spines": additional_file},
                "",
            )
        )

def build_segmentation_mesh(filename: str, target_dir: str, scale: list, min_component_rate = MIN_COMPONENT_SIZE_RATE):
    segm_image = imread(filename).astype(np.float32)
    img_min = segm_image.min()
    segmentation = np.round((segm_image - img_min) / (segm_image.max() - img_min) * 2).astype(np.uint8)
    min_component_size = int(np.prod(segmentation.shape)) * min_component_rate

    components = extract_large_components(
        segmentation,
        min_component_size=min_component_size,
    )
    n_comp = len(components)

    if n_comp == 0:
        raise ValueError("No valid connected components found")
    
    print(f"Найдено {n_comp} компонент сегментации. будет построено {n_comp} участков дендрита ({filename})", flush=True)

    results = []

    for i, component in enumerate(components):

        postfix = f"_{i}" if len(components) > 1 else ""
        folder = target_dir + postfix
        os.makedirs(folder, exist_ok=True)

        result = reconstruct_surface(
            component,
            component.shape,
            scale,
            folder
        )

        if result is not None:
            results.append(result)
        
        print(f"завершена обработка  {i} компоненты сегментации. ({filename})", flush=True)

    return results

# Обработка сегментации для нескольких снимков сразу

#### подготовка
- Сегментации полученные моделью собрать в одну директорию. 
- В директории не должно быть других файлов помимо файлов сегментации с расширением .tif. 
- Файлы сегментации должны быть в виде масок, где воксели, принадлежащие телу дендрита, отмечены значением 1, а воксели, принадлежащие шипикам, отмечены значечениями больше 1
- Собранные сегментации должны иметь единый размер воксельной сетки в микронах, он указывается для всей директории. иначе стоит собрать несколько директорий, запуская их обработку отдельно

#### запуск
1. в переменную scale запишите размер воксельной сетки в микронах, как [z_size, x_size, y_size]
2. в переменной dataset_path в строке указать путь до подготовленной директории
3. выполнить код в блоке ниже
4. в указанной директории для каждого из снимков сегментации появится директория с именем, соответствующем имени файла сегментации, с результатом построения меша


In [ ]:
from pathlib import Path

scale = [0.1, 0.025, 0.025]
dataset_path = Path(
    r"H:\Programs\Pyton\segmentation_test_dataset\ab_spinetool2.0 inference"
)


def process_file(entry, scale):
    print(entry)
    try:
        entry = Path(entry)
        print(
            "Найден файл сегментации:",
            entry.absolute(),
            flush=True
        )

        folder = str(entry.with_suffix("").absolute())

        build_segmentation_mesh(
            str(entry.absolute()),
            folder,
            scale,
        )

        print(f"[DONE] {entry.name}", flush=True)

    except Exception as e:
        print(f"[ERROR] {entry.name}: {e}", flush=True)



tif_files = [
        str(entry)
        for entry in dataset_path.iterdir()
        if entry.is_file() and entry.suffix.lower() == ".tif"
    ]

for file in tif_files:
    process_file(file, scale)

# Обработка сегментации для 1 снимка

#### подготовка
- Файл сегментации должен быть в виде маски, где воксели, принадлежащие телу дендрита, отмечены значением 1, а воксели, принадлежащие шипикам, отмечены значечениями больше 1

#### запуск
1. в переменную scale запишите размер воксельной сетки в микронах, как [z_size, x_size, y_size]
2. в переменной segmentation_path в строке указать путь до .tif файла сегментации
3. выполнить код в блоке ниже
4. рядом с указанным файлом появится директория с именем, соответствующем имени файла сегментации, с результатом построения меша
5. если на изображении найдено несколько дендритных стволов, для каждого отдельного будет создана директория, тогда имя будет содержить постфикс _i по номеру дендритного ствола


In [ ]:
scale = [0.1, 0.025, 0.025]
segmentation_path = Path(r"H:\Programs\Pyton\spinetool20_projects\datasets\wt7-12\wt_10_final.tif")

folder = str(segmentation_path.with_suffix('').absolute())
build_segmentation_mesh(str(segmentation_path.absolute()), folder, scale)

# Утилиты просмотра результата

#### подготовка запуска
1. в переменной segmentation_path в строке указать путь до .tif файла сегментации
2. если требуется визуализация меша, в переменной mesh_folder в строке указать путь до директории с результатами сегментации. Иначе оставить None

In [ ]:
segmentation_path = Path(r"H:\Programs\Pyton\segmentation_test_dataset\ab_spinetool2.0 inference\ab_20_neurosegm_0\ab_20_neurosegm.tif")
mesh_folder = None 
# пример заполнения
mesh_folder = Path(r"H:\Programs\Pyton\segmentation_test_dataset\ab_spinetool2.0 inference\ab_20_neurosegm_0")

#### Визуализация воксельной маски сегментации

1. получение проекции максимальной интенсивности

In [ ]:
segmentation = imread(segmentation_path.absolute()).astype(np.uint8)
imshow(segmentation.max(axis=0)[::-1, ...])

2. получение i-го z среза маски сегментации. Параметр z регулируется по желанию от 0 до максимального z для воксельной маски

In [ ]:
z = 20
segmentation = imread(segmentation_path.absolute()).astype(np.uint8)

if z > segmentation.shape[0] - 1:
    z = segmentation.shape[0] - 1
    print(f"Z слишком большой. уменьшен до {z}")

imshow(segmentation[z, ::-1, ...])

#### Визуализация меша сегментации

1. получение меша дендрита

In [ ]:
import trimesh

if mesh_folder is None:
    raise ValueError("Не указан путь к мешу для построения, вернитесь на шаг настройки запуска визуализации и укажите путь")

folder_str = str(mesh_folder)

mesh_file = "surface_mesh.off"
scene = trimesh.Scene()
surf: trimesh.Trimesh = trimesh.load_mesh(folder_str + f"\{mesh_file}")
scene.add_geometry(surf)
scene.show()

2. получение мешей шипиков отдельно от ствола дендрита

In [ ]:
import re

from matplotlib import pyplot as plt
import trimesh

if mesh_folder is None:
    raise ValueError("Не указан путь к мешу для построения, вернитесь на шаг настройки запуска визуализации и укажите путь")

folder_str = str(mesh_folder)

mesh_file = "surface_mesh.off"
pattern = re.compile(r"spine_(\d+)\.off")
spine_data = []

pattern = re.compile(r"spine_(\d+)\.off")

for filename in os.listdir(folder_str):
    match = pattern.search(filename)
    if match:
        index = int(match.group(1))
        full_path = os.path.join(folder_str, filename)
        
        spine_data.append(full_path)


cmap = plt.get_cmap('gist_rainbow')
scene = trimesh.Scene()
n_spines = len(spine_data)

for i, spine_path in enumerate(spine_data):
    mesh0: trimesh.Trimesh = trimesh.load_mesh(folder_str + f"\spine_{i}.off")
    color = (np.array(cmap(i / n_spines)) * 255).astype(np.uint8)
    mesh0.visual.face_colors = color
    scene.add_geometry(mesh0)
scene.show()

3. визуализация сегментации: вид сегментированных шипиков относительно ствола дендрита

In [ ]:
import re

from matplotlib import pyplot as plt
import trimesh

if mesh_folder is None:
    raise ValueError("Не указан путь к мешу для построения, вернитесь на шаг настройки запуска визуализации и укажите путь")

folder_str = str(mesh_folder)

mesh_file = "surface_mesh.off"
pattern = re.compile(r"spine_(\d+)\.off")
spine_data = []

pattern = re.compile(r"spine_(\d+)\.off")

for filename in os.listdir(folder_str):
    match = pattern.search(filename)
    if match:
        index = int(match.group(1))
        full_path = os.path.join(folder_str, filename)
        
        spine_data.append(full_path)


cmap = plt.get_cmap('gist_rainbow')
scene = trimesh.Scene()
surf: trimesh.Trimesh = trimesh.load_mesh(folder_str + f"\{mesh_file}")
scene.add_geometry(surf)
n_spines = len(spine_data)

for i, spine_path in enumerate(spine_data):
    mesh0: trimesh.Trimesh = trimesh.load_mesh(spine_path)
    color = (np.array(cmap(i / n_spines)) * 255).astype(np.uint8)
    mesh0.visual.face_colors = color
    scene.add_geometry(mesh0)
scene.show()

# Просмотр дендритного шипика на меше

1. укажите spine_number номер дендритного шипика (из имени файла), который хотите визуализировать
2. если дендритный шипик не является шипиком - удалите его из директории, чтобы удалить шипик из сегментации

In [ ]:
import trimesh

spine_number = 10


if mesh_folder is None:
    raise ValueError("Не указан путь к мешу для построения, вернитесь на шаг настройки запуска визуализации и укажите путь")

folder_str = str(mesh_folder)

mesh_file = "surface_mesh.off"
spine_file = f"spine_{spine_number}.off"

scene = trimesh.Scene()
surf: trimesh.Trimesh = trimesh.load_mesh(folder_str + f"\{mesh_file}")
scene.add_geometry(surf)
mesh0: trimesh.Trimesh = trimesh.load_mesh(folder_str + f"\{spine_file}")
color = [255, 0, 0, 255] # Red
mesh0.visual.face_colors = color
scene.add_geometry(mesh0)
scene.show()